# LeetCode 1167 — Minimum Cost to Connect Sticks bằng Python

Link đề: https://leetcode.com/problems/minimum-cost-to-connect-sticks/ · Độ khó: **Medium** · Chủ đề: Array, Greedy, Heap (Priority Queue)

> **Bài Premium** — đề viết theo hiểu biết về bài (và bản mô tả công khai trên kho tham khảo doocs), nên đối chiếu lại nếu có tài khoản Premium. Chữ ký hàm dưới đây suy ra từ lời giải Python công khai.

Skeleton LeetCode:

```python
class Solution:
    def connectSticks(self, sticks: List[int]) -> int:
        ...
```

Ghi chú: đáp án là **một số nguyên** — tổng chi phí nhỏ nhất để nối mọi que thành một, mỗi lần nối hai que dài `x`, `y` tốn `x + y`. Điểm cốt lõi: **tham lam** — luôn nối **hai que ngắn nhất** hiện có, dùng **min-heap** (hàng đợi ưu tiên). Đây chính là thuật toán xây **mã Huffman**.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chi phí mỗi lần nối = độ dài que mới, que mới tiếp tục tham gia các lần nối sau.
2. Nắm công cụ: `heapq` (min-heap: `heapify`, `heappop`, `heappush`, `heapreplace`), `bisect.insort`, `collections.deque`.
3. Đi từ **vét cạn mọi thứ tự nối** → **sort lại mỗi bước** → **danh sách đã sắp + insort** → **min-heap** → **hai hàng đợi** (sau khi sort).
4. Hiểu vì sao tham lam đúng: mỗi que bị "cộng" vào chi phí bằng **độ sâu** của nó trong cây nối → que dài nên ở nông, que ngắn ở sâu (lập luận Huffman / exchange argument).
5. Tránh lỗi: nối theo thứ tự đầu vào, sort một lần rồi cộng dồn trái → phải, quên trường hợp chỉ một que.

## 2. Đề bài & ví dụ

Bạn có một số que với độ dài nguyên dương, cho bởi mảng `sticks`. Có thể nối **hai que bất kỳ** dài `x` và `y` thành một que dài `x + y`, trả chi phí `x + y`. Phải nối cho đến khi chỉ còn **một** que.

Trả về **tổng chi phí nhỏ nhất**.

**Ví dụ 1:** `sticks = [2,4,3]` → `14`. Nối `2+3=5` (chi phí 5), rồi `5+4=9` (chi phí 9). Tổng `14`.

**Ví dụ 2:** `sticks = [1,8,3,5]` → `30`. `1+3=4`, `4+5=9`, `9+8=17` → `4+9+17 = 30`.

**Ví dụ 3:** `sticks = [5]` → `0`. Chỉ một que, không cần nối.

**Ràng buộc:** `1 <= sticks.length <= 10^4`; `1 <= sticks[i] <= 10^4`.

In [1]:
import bisect
import heapq
import random
import time
from collections import deque
from functools import lru_cache
from typing import List

examples = [
    ([2, 4, 3], 14),
    ([1, 8, 3, 5], 30),
    ([5], 0),
]
for s, want in examples:
    print(s, "->", want)

[2, 4, 3] -> 14
[1, 8, 3, 5] -> 30
[5] -> 0


## 3. Kiến thức nền

### 3.1 Min-heap với `heapq`

**Heap** (đống) là cây nhị phân gần đầy đủ lưu trong mảng, thoả: cha ≤ con (min-heap). Phần tử nhỏ nhất luôn ở `h[0]`. Các thao tác:

- `heapq.heapify(a)`: biến list thành heap tại chỗ, `O(n)`.
- `heapq.heappop(h)`: lấy và xoá phần tử nhỏ nhất, `O(log n)`.
- `heapq.heappush(h, x)`: thêm, `O(log n)`.
- `heapq.heapreplace(h, x)`: pop rồi push trong một bước (nhanh hơn).

In [2]:
h = [5, 1, 8, 3]
heapq.heapify(h)
print("heap:", h, "min:", h[0])
a = heapq.heappop(h); b = heapq.heappop(h)
heapq.heappush(h, a + b)
print("pop", a, b, "push", a + b, "->", sorted(h))
assert (a, b) == (1, 3)

heap: [1, 3, 8, 5] min: 1
pop 1 3 push 4 -> [4, 5, 8]


### 3.2 `bisect.insort` — chèn vào list đã sắp

`bisect.insort(a, x)` tìm vị trí bằng tìm kiếm nhị phân `O(log n)` nhưng **chèn** vào list tốn `O(n)` (dịch phần tử). Hữu ích làm bước trung gian giữa "sort lại" và "heap".

In [3]:
a = [1, 3, 5, 8]
bisect.insort(a, 4)
print(a)
assert a == [1, 3, 4, 5, 8]

[1, 3, 4, 5, 8]


### 3.3 `deque` làm hàng đợi FIFO

`popleft()` `O(1)`. Dùng cho cách "hai hàng đợi" ở mục 7.

In [4]:
q = deque([1, 2, 3])
print(q.popleft(), q[0] if q else None)

1 2


### 3.4 Cây nối và "độ sâu"

Mỗi chuỗi nối tạo một **cây nhị phân**: lá là que ban đầu, mỗi nút trong là một lần nối (giá trị = tổng lá bên dưới). Chi phí = tổng các nút trong = `Σ len(lá) × depth(lá)` (độ sâu tính từ gốc, gốc = 0). Kiểm chứng trên ví dụ 2: `1` và `3` ở độ sâu 3, `5` độ sâu 2, `8` độ sâu 1 → `1·3 + 3·3 + 5·2 + 8·1 = 30`.

In [5]:
print(1 * 3 + 3 * 3 + 5 * 2 + 8 * 1)
assert 1 * 3 + 3 * 3 + 5 * 2 + 8 * 1 == 30

30


### 3.5 `lru_cache` — ghi nhớ kết quả hàm (memoization)

Dùng cho brute force: trạng thái là **multiset** các que (biểu diễn bằng tuple đã sort), tránh tính lại.

In [6]:
@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)
print(fib(50))

12586269025


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 2 `[1, 8, 3, 5]` với hai chiến lược:

**Nối theo thứ tự đầu vào (trái → phải):** `1+8=9` (9), `9+3=12` (12), `12+5=17` (17) → `38`.

**Luôn nối hai que ngắn nhất:** `1+3=4` (4) → `[4,8,5]`; `4+5=9` (9) → `[9,8]`; `9+8=17` (17) → `30`.

Lý do: que ngắn bị "cộng lại" nhiều lần (nó nằm trong mọi que lớn hơn được tạo từ nó). Hãy để **que dài tham gia muộn**, chỉ bị cộng ít lần. Lần nối **cuối cùng** luôn tốn đúng `sum(sticks)` dù làm thế nào — chỉ các lần nối trước mới khác nhau.

In [7]:
def cost_of_order(sticks):
    """Chi phí khi nối lần lượt trái → phải."""
    total, cur = 0, sticks[0]
    for x in sticks[1:]:
        cur += x
        total += cur
    return total

print("trái→phải:", cost_of_order([1, 8, 3, 5]), "| ngắn nhất trước:", 30)

trái→phải: 38 | ngắn nhất trước: 30


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử **mọi** cách chọn hai que để nối ở mỗi bước (đệ quy). Trạng thái = multiset độ dài que → biểu diễn bằng `tuple(sorted(...))` và ghi nhớ.

`best(state) = 0` nếu còn 1 que; ngược lại `min` qua mọi cặp `(i, j)` của `(a_i + a_j) + best(state mới)`.

In [8]:
def connect_brute(sticks):
    @lru_cache(maxsize=None)
    def best(state):
        if len(state) <= 1:
            return 0
        res = float("inf")
        n = len(state)
        for i in range(n):
            for j in range(i + 1, n):
                merged = state[i] + state[j]
                rest = state[:i] + state[i + 1:j] + state[j + 1:] + (merged,)
                res = min(res, merged + best(tuple(sorted(rest))))
        return res
    return best(tuple(sorted(sticks)))

for s, want in examples:
    got = connect_brute(s)
    print(s, got, want)
    assert got == want

[2, 4, 3] 14 14
[1, 8, 3, 5] 30 30
[5] 0 0


**Độ phức tạp:** số trạng thái bùng nổ theo cấp số mũ (mỗi bước `C(k, 2)` lựa chọn, `n − 1` bước → không memo cỡ `n! (n-1)! / 2^{n-1}`). Chỉ dùng được với `n ≤ 8`.

**Nút thắt:** ta thử mọi cặp, trong khi trực giác nói **chỉ cần cặp nhỏ nhất**. Cần chứng minh điều đó, rồi tìm cấu trúc dữ liệu lấy hai phần tử nhỏ nhất nhanh.

## 6. Bước đột phá

> **Insight (Huffman):** tồn tại một cách nối tối ưu mà **lần nối đầu tiên** là nối **hai que ngắn nhất**. Sau lần nối đó, bài toán còn lại có cùng dạng với `n − 1` que → áp dụng lặp lại.

Lập luận trao đổi (exchange argument) ngắn: chi phí = `Σ len × depth`. Trong cây tối ưu, xét hai lá **sâu nhất** là anh em (luôn có, vì nút trong sâu nhất có hai con là lá). Nếu hai que ngắn nhất `a ≤ b` không nằm ở đó, **đổi chỗ** chúng với hai lá sâu nhất: que ngắn xuống sâu hơn, que dài lên nông hơn → chi phí không tăng. Vậy có lời giải tối ưu với `a`, `b` là anh em ở đáy, tức là được nối với nhau — và có thể làm lần nối đó **đầu tiên**.

Quy tắc "nếu… thì…":

- Nếu còn ≥ 2 que → lấy hai que ngắn nhất `a`, `b`, cộng `a + b` vào tổng, đưa que `a + b` trở lại.
- Nếu còn 1 que → dừng.

Cấu trúc cần: lấy 2 phần tử nhỏ nhất + thêm phần tử mới, lặp `n − 1` lần → **min-heap**.

In [9]:
# Kiểm chứng insight: tham lam (sort lại mỗi bước) == brute force trên nhiều input nhỏ
def greedy_resort(sticks):
    a = list(sticks); total = 0
    while len(a) > 1:
        a.sort()
        m = a[0] + a[1]
        total += m
        a = a[2:] + [m]
    return total

rng = random.Random(0)
for _ in range(200):
    s = [rng.randint(1, 9) for _ in range(rng.randint(1, 6))]
    assert greedy_resort(s) == connect_brute(s)
print("tham lam == brute trên 200 input nhỏ")

tham lam == brute trên 200 input nhỏ


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam, sort lại mỗi bước

Đã viết ở trên (`greedy_resort`). Đúng nhưng mỗi bước sort `O(n log n)` → tổng `O(n² log n)`. Với `n = 10^4`: cỡ `10^9` — quá chậm.

### 7.2 Cách 3 — List đã sắp + `bisect.insort`

Sort một lần. Mỗi bước lấy hai phần tử đầu (dùng con trỏ thay vì `pop(0)`), chèn tổng vào đúng chỗ bằng `insort`. Chèn `O(n)` → tổng `O(n²)` nhưng hằng số nhỏ (memmove trong C), chạy được `n = 10^4`.

In [10]:
def connect_insort(sticks):
    a = sorted(sticks)
    i, total = 0, 0                     # a[i:] là các que còn lại (đã sắp)
    while len(a) - i > 1:
        m = a[i] + a[i + 1]
        total += m
        i += 2
        bisect.insort(a, m, lo=i)
    return total

for s, want in examples:
    assert connect_insort(s) == want
print("insort OK")

insort OK


**Độ phức tạp:** `O(n²)` tệ nhất (chèn), `O(n)` bộ nhớ. Lưu ý list không co lại (ta chỉ dịch con trỏ `i`), độ dài tăng từ `n` lên `2n − 1`.

### 7.3 Cách 4 — Min-heap (lời giải chuẩn)

In [11]:
def connect_heap(sticks):
    h = list(sticks)
    heapq.heapify(h)
    total = 0
    while len(h) > 1:
        a = heapq.heappop(h)
        b = heapq.heappop(h)
        total += a + b
        heapq.heappush(h, a + b)
    return total

for s, want in examples:
    assert connect_heap(s) == want
print("heap OK")

heap OK


**Độ phức tạp:** `heapify` `O(n)`, `n − 1` vòng, mỗi vòng 3 thao tác `O(log n)` → `O(n log n)`. Bộ nhớ `O(n)`.

Tối ưu nhỏ: thay `heappop` thứ hai + `heappush` bằng `heapreplace` (pop + push một lần sift):

In [12]:
def connect_heap_replace(sticks):
    h = list(sticks)
    heapq.heapify(h)
    total = 0
    while len(h) > 1:
        a = heapq.heappop(h)
        m = a + h[0]                     # h[0] là que nhỏ thứ hai
        total += m
        heapq.heapreplace(h, m)          # thay que nhỏ thứ hai bằng que mới
    return total

for s, want in examples:
    assert connect_heap_replace(s) == want
print("heapreplace OK")

heapreplace OK


### 7.4 Cách 5 — Hai hàng đợi (sort + gộp tuyến tính)

Quan sát: các que **mới tạo** có độ dài **không giảm** theo thời gian (mỗi que mới là tổng hai que nhỏ nhất hiện tại, và que nhỏ nhất hiện tại không giảm). Vậy chỉ cần:

- `q1`: que ban đầu đã sort.
- `q2`: que mới tạo, tự động đã sắp theo thứ tự thêm vào.

Mỗi lần lấy phần tử nhỏ nhất: so sánh đầu `q1` và đầu `q2`. Sau sort, phần còn lại `O(n)`.

In [13]:
def connect_two_queues(sticks):
    q1 = deque(sorted(sticks))
    q2 = deque()

    def pop_min():
        if not q2 or (q1 and q1[0] <= q2[0]):
            return q1.popleft()
        return q2.popleft()

    total = 0
    for _ in range(len(sticks) - 1):
        m = pop_min() + pop_min()
        total += m
        q2.append(m)
    return total

for s, want in examples:
    assert connect_two_queues(s) == want
print("two queues OK")

two queues OK


**Độ phức tạp:** `O(n log n)` cho sort (hoặc `O(n + V)` nếu dùng counting sort vì `sticks[i] ≤ 10^4`), phần gộp `O(n)`. Bộ nhớ `O(n)`.

## 8. TRACE từng bước

Trace heap trên `[1, 8, 3, 5]`:

| Bước | Heap trước (sắp xếp để dễ nhìn) | a | b | a+b | total |
|------|---------------------------------|---|---|-----|-------|
| 1 | [1, 3, 5, 8] | 1 | 3 | 4 | 4 |
| 2 | [4, 5, 8] | 4 | 5 | 9 | 13 |
| 3 | [8, 9] | 8 | 9 | 17 | 30 |

In [14]:
def trace_heap(sticks):
    h = list(sticks)
    heapq.heapify(h)
    total, step = 0, 0
    print("heapify ->", h)
    while len(h) > 1:
        step += 1
        before = sorted(h)
        a = heapq.heappop(h); b = heapq.heappop(h)
        total += a + b
        heapq.heappush(h, a + b)
        print(f"bước {step}: {before} pop {a},{b} -> push {a+b}; total={total}; heap(mảng)={h}")
    return total

assert trace_heap([1, 8, 3, 5]) == 30
print()
assert trace_heap([3, 3, 3, 3, 3]) == connect_brute([3, 3, 3, 3, 3])

heapify -> [1, 5, 3, 8]
bước 1: [1, 3, 5, 8] pop 1,3 -> push 4; total=4; heap(mảng)=[4, 8, 5]
bước 2: [4, 5, 8] pop 4,5 -> push 9; total=13; heap(mảng)=[8, 9]
bước 3: [8, 9] pop 8,9 -> push 17; total=30; heap(mảng)=[17]

heapify -> [3, 3, 3, 3, 3]
bước 1: [3, 3, 3, 3, 3] pop 3,3 -> push 6; total=6; heap(mảng)=[3, 3, 3, 6]
bước 2: [3, 3, 3, 6] pop 3,3 -> push 6; total=12; heap(mảng)=[3, 6, 6]
bước 3: [3, 6, 6] pop 3,6 -> push 9; total=21; heap(mảng)=[6, 9]
bước 4: [6, 9] pop 6,9 -> push 15; total=36; heap(mảng)=[15]


Trace hai hàng đợi trên `[2, 2, 3, 7, 9]` — để thấy `q2` luôn tăng dần.

In [15]:
def trace_two_queues(sticks):
    q1, q2 = deque(sorted(sticks)), deque()
    total = 0
    for step in range(1, len(sticks)):
        picks = []
        for _ in range(2):
            if not q2 or (q1 and q1[0] <= q2[0]):
                picks.append(("q1", q1.popleft()))
            else:
                picks.append(("q2", q2.popleft()))
        m = picks[0][1] + picks[1][1]
        total += m
        q2.append(m)
        print(f"bước {step}: lấy {picks} -> {m}; q1={list(q1)} q2={list(q2)} total={total}")
    return total

assert trace_two_queues([2, 2, 3, 7, 9]) == connect_heap([2, 2, 3, 7, 9])

bước 1: lấy [('q1', 2), ('q1', 2)] -> 4; q1=[3, 7, 9] q2=[4] total=4
bước 2: lấy [('q1', 3), ('q2', 4)] -> 7; q1=[7, 9] q2=[7] total=11
bước 3: lấy [('q1', 7), ('q2', 7)] -> 14; q1=[9] q2=[14] total=25
bước 4: lấy [('q1', 9), ('q2', 14)] -> 23; q1=[] q2=[23] total=48


## 9. Code nộp LeetCode

In [16]:
import heapq
from typing import List


class Solution:
    def connectSticks(self, sticks: List[int]) -> int:
        heapq.heapify(sticks)               # biến input thành min-heap tại chỗ
        total = 0
        while len(sticks) > 1:
            a = heapq.heappop(sticks)
            b = heapq.heappop(sticks)
            total += a + b
            heapq.heappush(sticks, a + b)
        return total


sol = Solution()
for s, want in examples:
    assert sol.connectSticks(list(s)) == want   # truyền bản sao vì hàm sửa input
print("Solution OK")

Solution OK


## 10. Vì sao đúng

**Bổ đề 1 (chi phí = tổng trọng số theo độ sâu).** Mỗi lần nối tạo một nút trong có giá trị = tổng các lá trong cây con. Tổng các nút trong = mỗi lá được đếm một lần cho mỗi tổ tiên = `Σ len(lá) · depth(lá)`.

**Bổ đề 2 (lựa chọn tham lam an toàn).** Gọi `a ≤ b` là hai que ngắn nhất. Trong một cây tối ưu `T`, lấy nút trong sâu nhất; hai con `x, y` của nó là lá ở độ sâu lớn nhất `D`. Đổi `a ↔ x`, `b ↔ y`: vì `a ≤ x`, `depth(a) ≤ D`, chi phí thay đổi `(x − a)(depth(a) − D) ≤ 0`; tương tự cho `b`. Cây mới vẫn tối ưu và `a, b` là anh em.

**Bổ đề 3 (cấu trúc con tối ưu).** Nếu `a, b` là anh em, thay chúng bằng một lá `a + b` cho cây `T'` của bài toán `n − 1` que, với `cost(T) = cost(T') + (a + b)`. Vậy tối ưu cho `n` que = `(a + b)` + tối ưu cho `n − 1` que (với que `a+b` thay hai que kia).

Quy nạp theo `n` → thuật toán tham lam (lặp lại bổ đề 2+3) cho chi phí tối ưu. Min-heap chỉ là cách cài đặt "lấy hai que ngắn nhất" hiệu quả.

**Vì sao hai hàng đợi đúng:** que mới thứ `k+1` = tổng hai que nhỏ nhất ở thời điểm `k+1`, các que này ≥ các que nhỏ nhất ở thời điểm `k` (vì đã lấy ra hai que nhỏ nhất và chỉ thêm que lớn hơn chúng) → dãy que mới không giảm.

## 11. Time / Space Complexity

Gọi `n = len(sticks)`, `V = max(sticks)`.

| Cách | Time | Space |
|------|------|-------|
| 1. Brute + memo | mũ | mũ |
| 2. Sort lại mỗi bước | `O(n² log n)` | `O(n)` |
| 3. insort | `O(n²)` (chèn), hằng số nhỏ | `O(n)` |
| 4. Min-heap | `O(n log n)` | `O(n)` (hoặc `O(1)` thêm nếu heapify tại chỗ) |
| 5. Hai hàng đợi | `O(n log n)` sort + `O(n)`; `O(n + V)` với counting sort | `O(n)` |

Giá trị lớn nhất của tổng: mỗi que dài ≤ `10^4`, tổng ≤ `10^8`, chi phí ≤ `n · tổng` ≈ `10^12` — Python int không tràn số, nhưng ở C++/Java cần `long`.

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [17]:
impls = {
    "resort": greedy_resort,
    "insort": connect_insort,
    "heap": connect_heap,
    "heapreplace": connect_heap_replace,
    "two_queues": connect_two_queues,
    "Solution": lambda s: Solution().connectSticks(list(s)),
}
edge = [
    ([1], 0),
    ([1, 1], 2),
    ([10000, 10000], 20000),
    ([1, 2, 3, 4, 5], 33),
    ([5, 5, 5, 5], 40),
    ([1, 1, 1, 1, 100], 0),   # đáp án điền bằng brute force ngay dưới
]
edge[-1] = ([1, 1, 1, 1, 100], connect_brute([1, 1, 1, 1, 100]))
for name, f in impls.items():
    for s, want in examples + edge:
        before = list(s)
        assert f(s) == want, (name, s)
        assert s == before, (name, "đã sửa input")
print("all impls pass", len(examples + edge))
print("[1,1,1,1,100] ->", edge[-1][1])

all impls pass 9
[1,1,1,1,100] -> 112


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

In [18]:
rng = random.Random(1167)
for _ in range(400):
    s = [rng.randint(1, 20) for _ in range(rng.randint(1, 7))]
    want = connect_brute(s)
    for name, f in impls.items():
        assert f(s) == want, (name, s)
print("random cross-check OK")

random cross-check OK


Đối chiếu các cách nhanh với nhau trên mảng vừa (brute quá chậm ở đây):

In [19]:
for _ in range(100):
    s = [rng.randint(1, 10**4) for _ in range(rng.randint(1, 300))]
    vals = {name: f(s) for name, f in impls.items()}
    assert len(set(vals.values())) == 1, vals
print("fast impls agree")

fast impls agree


### 12.3 Test giới hạn

`n = 10^4`, giá trị ngẫu nhiên và toàn bằng nhau.

In [20]:
big = [random.Random(9).randint(1, 10**4) for _ in range(10**4)]
for s in (big, [10**4] * 10**4, list(range(1, 10**4 + 1))):
    for name in ["heap", "two_queues", "Solution", "insort"]:
        t0 = time.perf_counter()
        r = impls[name](s)
        print(f"{name:11} {r} {time.perf_counter() - t0:.3f}s")
    print()

heap        1013610976 0.006s
two_queues  1013610976 0.003s
Solution    1013610976 0.007s
insort      1013610976 0.003s

heap        1336160000 0.005s
two_queues  1336160000 0.002s
Solution    1336160000 0.004s
insort      1336160000 0.002s

heap        652354680 0.005s
two_queues  652354680 0.002s
Solution    652354680 0.004s
insort      652354680 0.005s



## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Nối theo thứ tự đầu vào

In [21]:
def wrong_in_order(sticks):
    return cost_of_order(sticks) if len(sticks) > 1 else 0

s = [1, 8, 3, 5]
print("sai:", wrong_in_order(s), "| đúng:", connect_heap(s))
assert wrong_in_order(s) != connect_heap(s)

sai: 38 | đúng: 30


**Vì sao sai:** que dài `8` bị nối sớm và bị cộng lại nhiều lần. **Sửa:** luôn chọn hai que ngắn nhất.

### Lỗi 2 — Sort một lần rồi cộng dồn trái → phải

In [22]:
def wrong_sort_once(sticks):
    s = sorted(sticks)
    return cost_of_order(s) if len(s) > 1 else 0

s = [3, 3, 3, 3]
print("sai:", wrong_sort_once(s), "| đúng:", connect_heap(s))
assert wrong_sort_once(s) != connect_heap(s)

sai: 27 | đúng: 24


**Vì sao sai:** sau khi nối `3+3=6`, que `6` không còn là nhỏ nhất (còn hai que `3`). Cộng dồn ép que mới luôn tham gia lần nối kế tiếp. Đúng là `3+3`, `3+3`, `6+6` = `24`, sai cho `6+9+12 = 27`. **Sửa:** que mới phải quay lại "cuộc đua" (heap / hai hàng đợi).

### Lỗi 3 — Dùng max-heap hoặc pop nhầm phần tử lớn nhất

In [23]:
def wrong_largest_first(sticks):
    a = sorted(sticks); total = 0
    while len(a) > 1:
        m = a.pop() + a.pop()          # lấy hai que DÀI nhất
        total += m
        bisect.insort(a, m)
    return total

s = [1, 2, 3, 4]
print("sai:", wrong_largest_first(s), "| đúng:", connect_heap(s))
assert wrong_largest_first(s) > connect_heap(s)

sai: 26 | đúng: 19


**Vì sao sai:** ngược hoàn toàn với insight; `heapq` của Python là **min-heap**, đừng đổi dấu. **Sửa:** pop phần tử nhỏ nhất.

### Lỗi 4 — Chỉ dùng **một** hàng đợi: đưa que mới vào cuối hàng đợi đã sort

In [24]:
def wrong_single_queue(sticks):
    q = deque(sorted(sticks))
    total = 0
    while len(q) > 1:
        m = q.popleft() + q.popleft()
        total += m
        q.append(m)                     # que mới xếp cuối, dù có thể nhỏ hơn que đang chờ
    return total

s = [1, 2, 3, 4]
print("sai:", wrong_single_queue(s), "| đúng:", connect_heap(s))
assert wrong_single_queue(s) != connect_heap(s)

sai: 20 | đúng: 19


**Vì sao sai:** sau `1+2=3`, hàng đợi là `[3, 4, 3]` — que mới `3` đứng sau `4` nên lần kế tiếp nối `3+4` thay vì `3+3`. Kết quả `20` thay vì `19`. **Sửa:** dùng **hai** hàng đợi và so sánh đầu hai hàng (cách 5), hoặc dùng heap.

### Lỗi 5 — Pop hai lần khi chỉ còn một que

`while h:` thay vì `while len(h) > 1:` sẽ gọi `heappop` trên heap rỗng → `IndexError` với input `[5]`. Luôn kiểm tra điều kiện dừng là "còn ít nhất 2 que".

In [25]:
def wrong_loop_condition(sticks):
    h = list(sticks); heapq.heapify(h); total = 0
    while h:
        a = heapq.heappop(h)
        if not h:
            break                        # phải thêm nhánh thoát này mới không lỗi
        b = heapq.heappop(h)
        total += a + b
        heapq.heappush(h, a + b)
    return total

try:
    h = [5]; heapq.heappop(h); heapq.heappop(h)
except IndexError as e:
    print("IndexError:", e)
print(wrong_loop_condition([5]), wrong_loop_condition([1, 8, 3, 5]))

IndexError: index out of range
0 30


## 14. Mẹo & nhận dạng pattern

- **"Gộp hai phần tử, chi phí = tổng, lặp đến khi còn một"** → Huffman: min-heap, luôn gộp hai phần tử nhỏ nhất.
- **Chi phí phụ thuộc "số lần một phần tử bị cộng lại"** → nghĩ theo cây và độ sâu; phần tử lớn nên ở nông.
- **Liên tục lấy min và thêm phần tử mới** → `heapq`. Nếu phần tử mới tạo ra **đơn điệu** → có thể thay heap bằng hai hàng đợi.
- **Khác với gộp kề nhau** (chỉ được gộp hai đống **liền kề** — LC 1000 Minimum Cost to Merge Stones): khi đó tham lam sai, phải dùng DP khoảng.

## 15. Biến thể

1. **Mã Huffman** — cùng thuật toán, tần suất ký tự thay cho độ dài que; độ sâu lá = độ dài mã.
2. **LC 1000 Minimum Cost to Merge Stones** — chỉ gộp `k` đống **liên tiếp** → DP khoảng `O(n³/k)`.
3. **Gộp `k` que một lần (k-ary Huffman)** — thêm que độ dài `0` cho đến khi `(n − 1) % (k − 1) == 0`, rồi mỗi lần gộp `k` que nhỏ nhất.
4. **LC 2208 Minimum Operations to Halve Array Sum** — max-heap tham lam.
5. **Merge `k` file đã sort** (Optimal Merge Pattern) — giống hệt bài này.

In [26]:
# Biến thể 3: mỗi lần gộp k que (k >= 2)
def connect_k(sticks, k):
    h = list(sticks)
    while (len(h) - 1) % (k - 1):
        h.append(0)
    heapq.heapify(h)
    total = 0
    while len(h) > 1:
        m = sum(heapq.heappop(h) for _ in range(k))
        total += m
        heapq.heappush(h, m)
    return total

print(connect_k([1, 8, 3, 5], 2), connect_k([1, 2, 3, 4, 5], 3))
assert connect_k([1, 8, 3, 5], 2) == 30
assert connect_k([1, 2, 3, 4, 5], 3) == 6 + 15

30 21


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Brute + memo | Thử mọi cặp mỗi bước | mũ | mũ | Kiểm chứng `n ≤ 7` |
| 2. Sort lại | Tham lam ngây thơ | `O(n² log n)` | `O(n)` | Minh hoạ insight |
| 3. insort | List đã sắp, chèn nhị phân | `O(n²)` | `O(n)` | Khi chưa biết heap |
| 4. Min-heap | Tham lam + heap | `O(n log n)` | `O(n)` | **Nộp bài** |
| 5. Hai hàng đợi | Que mới tăng dần | `O(n log n)` / `O(n + V)` | `O(n)` | Input đã sort hoặc giá trị nhỏ |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** `connectSticks([1, 1, 1, 1])` = ?

**Bài 2 (tính tay):** trace heap cho `[4, 1, 6, 2, 9]`, ghi từng bước và tổng.

**Bài 3 (điền TODO):** viết hàm trả về **chuỗi các lần nối** `[(a, b), ...]` theo thứ tự tham lam.

**Bài 4 (tự viết lại):** viết lại cách 5 nhưng dùng **counting sort** (vì `sticks[i] ≤ 10^4`) thay `sorted`.

**Bài 5 (mini challenge):** tính **độ dài mã Huffman** của mỗi ký tự cho bảng tần suất `{'a': 5, 'b': 9, 'c': 12, 'd': 13, 'e': 16, 'f': 45}` và kiểm chứng `Σ freq × len = connectSticks(freqs)`.

In [27]:
# Bài 3 — TODO
def merge_sequence(sticks):
    # TODO: trả về list các cặp (a, b) được nối, theo thứ tự
    raise NotImplementedError

# assert merge_sequence([1, 8, 3, 5]) == [(1, 3), (4, 5), (8, 9)]

In [28]:
# Bài 5 — TODO
def huffman_lengths(freq):
    # TODO: trả về dict ký tự -> độ dài mã
    pass

# L = huffman_lengths({'a': 5, 'b': 9, 'c': 12, 'd': 13, 'e': 16, 'f': 45})
# assert L['f'] == 1

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: `1+1`, `1+1`. Mức 2: còn `[2, 2]`. Mức 3: `2 + 2 + 4 = 8`.
- **Bài 2:** Mức 1: heap đầu `[1,2,4,6,9]`. Mức 2: các que mới `3, 7, 13, 22`. Mức 3: tổng `45`.
- **Bài 3:** Mức 1: copy cách 4. Mức 2: `seq.append((a, b))`. Mức 3: `a ≤ b` vì pop từ min-heap.
- **Bài 4:** Mức 1: mảng `cnt[0..10^4]`. Mức 2: duyệt `v` tăng dần, thêm `v` vào `q1` `cnt[v]` lần. Mức 3: giữ nguyên phần hai hàng đợi.
- **Bài 5:** Mức 1: heap chứa `(freq, id, danh sách ký tự)`. Mức 2: mỗi lần gộp, tăng độ dài của mọi ký tự trong hai nhóm thêm 1. Mức 3: `id` để tránh so sánh list khi hoà tần suất.

## 19. Đáp án tham khảo

In [29]:
# Bài 1
assert Solution().connectSticks([1, 1, 1, 1]) == 8
# Bài 2
assert trace_heap([4, 1, 6, 2, 9]) == 45

# Bài 3
def merge_sequence_sol(sticks):
    h = list(sticks); heapq.heapify(h); seq = []
    while len(h) > 1:
        a = heapq.heappop(h); b = heapq.heappop(h)
        seq.append((a, b)); heapq.heappush(h, a + b)
    return seq

assert merge_sequence_sol([1, 8, 3, 5]) == [(1, 3), (4, 5), (8, 9)]

# Bài 4
def connect_counting(sticks):
    cnt = [0] * (max(sticks) + 1)
    for x in sticks:
        cnt[x] += 1
    q1 = deque(v for v in range(len(cnt)) for _ in range(cnt[v]))
    q2 = deque(); total = 0
    def pop_min():
        if not q2 or (q1 and q1[0] <= q2[0]):
            return q1.popleft()
        return q2.popleft()
    for _ in range(len(sticks) - 1):
        m = pop_min() + pop_min(); total += m; q2.append(m)
    return total

rng = random.Random(17)
for _ in range(200):
    s = [rng.randint(1, 50) for _ in range(rng.randint(1, 40))]
    assert connect_counting(s) == connect_heap(s)
print("bài 1-4 OK")

heapify -> [1, 2, 6, 4, 9]
bước 1: [1, 2, 4, 6, 9] pop 1,2 -> push 3; total=3; heap(mảng)=[3, 4, 6, 9]
bước 2: [3, 4, 6, 9] pop 3,4 -> push 7; total=10; heap(mảng)=[6, 9, 7]
bước 3: [6, 7, 9] pop 6,7 -> push 13; total=23; heap(mảng)=[9, 13]
bước 4: [9, 13] pop 9,13 -> push 22; total=45; heap(mảng)=[22]
bài 1-4 OK


In [30]:
# Bài 5
def huffman_lengths_sol(freq):
    h = [(f, i, [ch]) for i, (ch, f) in enumerate(freq.items())]
    heapq.heapify(h)
    length = {ch: 0 for ch in freq}
    uid = len(h)
    while len(h) > 1:
        f1, _, g1 = heapq.heappop(h)
        f2, _, g2 = heapq.heappop(h)
        for ch in g1 + g2:
            length[ch] += 1
        heapq.heappush(h, (f1 + f2, uid, g1 + g2)); uid += 1
    return length

freq = {'a': 5, 'b': 9, 'c': 12, 'd': 13, 'e': 16, 'f': 45}
L = huffman_lengths_sol(freq)
print(L)
assert L['f'] == 1
assert sum(freq[c] * L[c] for c in freq) == Solution().connectSticks(list(freq.values()))
print("bài 5 OK")

{'a': 4, 'b': 4, 'c': 3, 'd': 3, 'e': 3, 'f': 1}
bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
heapq.heapify(sticks)
total = 0
while len(sticks) > 1:
    a = heapq.heappop(sticks)
    b = heapq.heappop(sticks)
    total += a + b
    heapq.heappush(sticks, a + b)
return total
```

**Checklist:**

- [ ] Điều kiện vòng lặp `len(h) > 1` (một que → `0`)?
- [ ] Dùng **min**-heap (lấy hai que ngắn nhất)?
- [ ] Que mới được **đưa lại** vào heap?
- [ ] Không nhầm với bài chỉ gộp phần tử kề nhau (khi đó tham lam sai)?
- [ ] Ngôn ngữ khác Python: chi phí có thể tới ~`10^12`, dùng kiểu 64-bit.